<a href="https://colab.research.google.com/github/chanchalr04/CodeAlpha-Internship/blob/main/Music_genration_with_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install tensorflow keras numpy mido pretty_midi pygame


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 33.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 720.9 kB/s eta 0:00:00
  Created wheel for pretty_midi: filename=pretty_midi-0.2.10-py3-none-any.whl size=5592287 sha256=c38ade3daeab25b623833a2637514d8d817dc07cc9f0fee3b315793b242ce1d7
  Stored in directory: /root/.cache/pip/wheels/e6/95/ac/15ceaeb2823b04d8e638fd1495357adb8d26c00ccac9d7782e
Successfully built pretty_midi


In [2]:
from google.colab import files

print("Please upload a file...")
uploaded_files = files.upload()
print("Upload successful!")


Please upload a file...


Saving baby.mid to baby.mid
Upload successful!


In [3]:
import mido
import numpy as np

def midi_to_notes(midi_file):
    mid = mido.MidiFile(midi_file)
    notes = []

    for msg in mid:
        if msg.type == 'note_on':
            notes.append(msg.note)

    return np.array(notes)

# Testing with a sample MIDI file
notes = midi_to_notes("/content/baby.mid")
print(notes)


[57 60 65 ... 38 42 42]


In [4]:
from tensorflow.keras.utils import to_categorical

sequence_length = 20  # Each input sequence will have 20 notes

# Creating input-output pairs
X, y = [], []
for i in range(len(notes) - sequence_length):
    X.append(notes[i:i + sequence_length])
    y.append(notes[i + sequence_length])

# Converting to NumPy arrays
X, y = np.array(X), np.array(y)

# Normalizing and reshaping input
X = X / 127.0
X = np.expand_dims(X, axis=-1)

# One-hot encoding the output (MIDI has 128 possible notes)
y = to_categorical(y, num_classes=128)


In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Creating the model
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(sequence_length, 1)),
    LSTM(128),
    Dense(128, activation='softmax')
])

# Compiling the model
model.compile(loss='categorical_crossentropy', optimizer='adam')

# Displaying the model summary
model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 20, 128)             │          66,560 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 128)                 │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │          16,512 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 214,656 (838.50 KB)

 Trainable params: 214,656 (838.50 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.fit(X, y, epochs=50, batch_size=64)  # Adjust epochs as needed


Epoch 1/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 19s 121ms/step - loss: 3.7226
Epoch 2/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 16s 80ms/step - loss: 3.0455
Epoch 3/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 9s 72ms/step - loss: 3.0532
Epoch 4/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 9s 84ms/step - loss: 3.0512
Epoch 5/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 10s 85ms/step - loss: 3.0441
Epoch 6/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 10s 79ms/step - loss: 3.0298
Epoch 7/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 10s 72ms/step - loss: 3.0387
Epoch 8/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 10s 73ms/step - loss: 3.0177
Epoch 9/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 11s 83ms/step - loss: 3.0403
Epoch 10/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 11s 89ms/step - loss: 3.0610
Epoch 11/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 10s 89ms/step - loss: 3.0473
Epoch 12/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 9s 74ms/step - loss: 3.0234
Epoch 13/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 10s 73ms/step - loss: 3.0390
Epoch 14/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 11s 85ms/step - loss: 3.0407
Epoch 15/50
101/101 ━━━━━━━━━━━

In [14]:
import random

def generate_music(model, seed_notes, length=100):
    output = list(seed_notes)  # Starting with the given seed notes

    for _ in range(length):  # Generating 100 notes
        input_seq = np.array(output[-sequence_length:]) / 127.0  # Normalizing
        input_seq = np.expand_dims(input_seq, axis=0)  # Reshaping for LSTM

        prediction = model.predict(input_seq)  # Predicting the next note
        next_note = np.argmax(prediction)  # Picking the most probable note

        output.append(next_note)  # Adding the note to the sequence

    return output

# Generating music with a random seed
seed_notes = [random.randint(40, 80) for _ in range(sequence_length)]  # Random seed notes
generated_notes = generate_music(model, seed_notes, length=100)
print(generated_notes)  # Printing generated notes


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━

In [8]:
from mido import MidiFile, MidiTrack, Message

def notes_to_midi(notes, filename="output.mid"):
    mid = MidiFile()  # Creating a new MIDI file
    track = MidiTrack()  # Adding a track
    mid.tracks.append(track)

    for note in notes:
        track.append(Message('note_on', note=note, velocity=64, time=120))  # Note on
        track.append(Message('note_off', note=note, velocity=64, time=120))  # Note off

    mid.save(filename)  # Saving the MIDI file
    print(f"Saved as {filename}")  # Confirmation message

notes_to_midi(generated_notes)  # Converting generated notes into a MIDI file


Saved as output.mid


In [15]:
import os
os.environ["SDL_AUDIODRIVER"] = "dummy"  # Headless mode
import pygame

pygame.init()
pygame.mixer.init()
print(pygame.mixer.get_init())  # Should print some audio settings


(44100, -16, 2)


In [16]:
import pygame
pygame.init()
print(pygame.mixer.get_init())


(44100, -16, 2)


In [17]:
!apt-get install fluidsynth


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fluidsynth is already the newest version (2.2.5-1).
0 upgraded, 0 newly installed, 0 to remove and 29 not upgraded.


In [18]:
!fluidsynth -ni soundfont.sf2 output.mid -F output.wav -r 44100


FluidSynth runtime version 2.2.5
Copyright (C) 2000-2022 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

fluidsynth: error: fluid_is_soundfont(): fopen() failed: 'File does not exist.'
Parameter 'soundfont.sf2' not a SoundFont or MIDI file or error occurred identifying it.
Rendering audio to file 'output.wav'..


In [19]:
from IPython.display import Audio
Audio("output.wav")